# One-Class SVM Anomaly Detection for TLS Profiling

This notebook evaluates one-class SVM baselines on extracted TLS traffic features. For each target label, the detector is trained only on flows from that label and then evaluated on its ability to separate that label from the rest of the traffic using the negated decision function as the anomaly score.


In [1]:
import sys
sys.path.append('../../src')


## Feature Groups

- `FLOW`: basic bidirectional flow statistics such as bytes, packets, rates, and duration (`bs`, `ps`, `br`, `pr`, `td`)
- `CTLS`: client-side TLS metadata (`tls.cver`, `tls.ccs`, `tls.cext`, `tls.csg`, `tls.alpn`, `tls.csv`)
- `STLS`: server-side TLS metadata (`tls.sver`, `tls.scs`, `tls.sext`, `tls.ssv`)
- `REC`: ordered sequence of signed TLS record lengths (`tls.rec`, first 20 records)

The notebook evaluates each group individually and all group combinations to understand which feature families carry the most discriminative signal for the one-class SVM baseline.


In [2]:
FLOW = ["bs","ps", "br", "pr", "td"]
CTLS = ["tls.cver", "tls.ccs", "tls.cext", "tls.csg", "tls.alpn", "tls.csv"]
STLS = ["tls.sver", "tls.scs", "tls.sext", "tls.ssv"]
REC = ["tls.rec"]


In [ ]:
from pathlib import Path

import pandas as pd
from tls_profiling.preprocessing import build_and_fit_preprocessor

data_dir = Path("../data-ext")
#data_dir = Path("../data")

print(f"Loading extracted features from {data_dir}.")
df_train = pd.read_parquet(data_dir / "malware_train.parquet")
df_val = pd.read_parquet(data_dir / "malware_val.parquet")
df_test = pd.read_parquet(data_dir / "malware_test.parquet")
df_train_label = pd.read_parquet(data_dir / "malware_train_labels.parquet")
df_val_label = pd.read_parquet(data_dir / "malware_val_labels.parquet")
df_test_label = pd.read_parquet(data_dir / "malware_test_labels.parquet")

preprocessor = build_and_fit_preprocessor(df_train)
print("Built preprocessor from df_train.")


In [ ]:
from tls_profiling.baselines.model_ocsvm import OneClassSVMDetector, Config
import numpy as np

OCSVM_CONFIG = Config(
    kernel="rbf",
    gamma="scale",
    nu=0.05,
    cache_size=512,
    max_train_samples=5_000,
)

def train_detector(train: np.ndarray) -> OneClassSVMDetector:
    detector = OneClassSVMDetector(OCSVM_CONFIG)
    detector.fit(train)
    return detector

from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve

import warnings

warnings.filterwarnings(
    "ignore",
    message=r"unknown class\(es\) .* will be ignored",
    category=UserWarning,
    module=r"sklearn\.preprocessing\._label",
)

def select_feature_set(x, feature_set):
    if not feature_set:
        return x

    selected_columns = [
        column for column in x.columns
        if any(column.startswith(prefix) for prefix in feature_set)
    ]
    return x.loc[:, selected_columns]

def choose_f1_threshold(y_true, anomaly_score):
    precision, recall, thresholds = precision_recall_curve(y_true, anomaly_score)

    if len(thresholds) == 0:
        return float("inf")

    f1_scores = (2 * precision[:-1] * recall[:-1]) / np.clip(
        precision[:-1] + recall[:-1],
        a_min=1e-12,
        a_max=None,
    )
    best_idx = int(np.nanargmax(f1_scores))
    return float(thresholds[best_idx])

def evaluate(feature_set, normal_label):
    # x_train: only normal traffic
    x_train = df_train.loc[df_train_label["category"] == normal_label].reset_index(drop=True)
    # x_val: mixed traffic used to tune the F1 threshold
    x_val = df_val
    y_val = (df_val_label["category"] != normal_label).astype(int)
    # y_test: 1 = anomaly, 0 = normal
    y_test = (df_test_label["category"] != normal_label).astype(int)
    x_test = df_test

    # prepare datasets
    x_train_transformed = select_feature_set(preprocessor.transform(x_train), feature_set)
    x_val_transformed = select_feature_set(preprocessor.transform(x_val), feature_set)
    x_test_transformed = select_feature_set(preprocessor.transform(x_test), feature_set)

    # create detector on TRAIN
    detector = train_detector(x_train_transformed)

    # choose the F1 threshold on the mixed validation split
    val_scores = detector.score(x_val_transformed)
    threshold = choose_f1_threshold(y_val, val_scores)

    # evaluate on TEST
    anomaly_score = detector.score(x_test_transformed)

    return y_test, anomaly_score, threshold

def debug_csv(feature_set, normal_label, y_test, y_pred, anomaly_score):
    """
    Write the intermediate data to CSV file.
    """
    feature_set_name = "_".join(feature_set)
    class_label_name = normal_label

    output_path = f"tmp/ocsvm_{class_label_name}_{feature_set_name}.csv"
    pd.DataFrame({
        "y_test": y_test,
        "y_pred": y_pred,
        "anomaly_score": anomaly_score,
    }).to_csv(output_path, index=False)

def compute_scores(feature_set, normal_label):
    y_test, anomaly_score, threshold = evaluate(feature_set, normal_label)
    auc = roc_auc_score(y_test, anomaly_score)
    ap = average_precision_score(y_test, anomaly_score)
    y_pred = (anomaly_score >= threshold).astype(int)
    f1 = f1_score(y_test, y_pred)

    debug_csv(feature_set, normal_label, y_test, y_pred, anomaly_score)
    return {"auc_score": auc, "ap_score": ap, "f1_score": f1}

def plot_curves(feature_set, normal_label):
    y_test, anomaly_score, threshold = evaluate(feature_set, normal_label)

    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    PrecisionRecallDisplay.from_predictions(
        y_test,
        anomaly_score,
        name="OneClassSVM PR-AUC",
        ax=axes[0],
    )
    axes[0].set_title(f"{normal_label} Precision-Recall")

    RocCurveDisplay.from_predictions(
        y_test,
        anomaly_score,
        name="OneClassSVM AUC-ROC",
        ax=axes[1],
    )
    axes[1].set_title(f"{normal_label} ROC Curve")

    plt.tight_layout()
    plt.show()


## Evaluation

For each label in `system`, `unknown`, and `malware`, that label is treated as the in-profile or normal class. All remaining labels are treated as anomalies.

The evaluation uses three disjoint splits:

- `train`: only samples from the selected normal class, used to fit the one-class SVM detector
- `validation`: mixed traffic, used to choose the anomaly-score threshold that maximizes `F1`
- `test`: held-out mixed traffic, used only for final reporting

Feature ablation:

- each feature group is evaluated individually
- all pairwise, triple, and full combinations are also evaluated

Reported metrics:

- `AUC-ROC`: how well the anomaly score ranks the selected class against the rest over all thresholds
- `AP`: average precision, which is especially useful when the class balance is uneven
- `F1`: test-set score obtained with the threshold selected on the validation split

Practical notes for this baseline:

- the detector uses an RBF-kernel one-class SVM with `gamma="scale"` and `nu=0.05`
- training is capped at `5,000` sampled in-profile flows per run because kernel one-class SVMs scale poorly with the full dataset size; the rest of the protocol stays aligned with the other baseline notebooks


In [5]:
from itertools import combinations

feature_groups = {
    "FLOW": FLOW,
    "CTLS": CTLS,
    "STLS": STLS,
    "REC": REC,
}

rows = []

group_names = list(feature_groups)
for size in range(1, len(group_names) + 1):
    for group_combo in combinations(group_names, size):
        selected_features = []
        for group_name in group_combo:
            selected_features.extend(feature_groups[group_name])

        feature_set_name = ", ".join(group_combo)

        for label in ["system", "unknown", "malware"]:
            scores = compute_scores(selected_features, label)

            rows.append({
                "FeatureSet": feature_set_name,
                "ClassLabel": label,
                "AucScore": scores["auc_score"],
                "ApScore": scores["ap_score"],
                "F1Score": scores["f1_score"],
            })

results_df = pd.DataFrame(rows, columns=["FeatureSet", "ClassLabel", "AucScore", "ApScore", "F1Score"])
display(results_df)


,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
0,FLOW,system,0.645345,0.802920,0.770835
1,FLOW,unknown,0.474697,0.842730,0.930261
2,FLOW,malware,0.663746,0.581082,0.754639
3,CTLS,system,0.849237,0.921399,0.882392
4,CTLS,unknown,0.491241,0.866406,0.928980
5,CTLS,malware,0.788850,0.810698,0.850664
6,STLS,system,0.594985,0.832080,0.725377
7,STLS,unknown,0.383855,0.853486,0.928980
8,STLS,malware,0.756999,0.821935,0.724171
9,REC,system,0.572438,0.755338,0.770829


## System Profile

The table below reports the one-class SVM baseline for the `system` profile across all feature-group combinations. Strong results here indicate that the selected features place system traffic inside a compact nonlinear region that the kernel boundary can represent reliably.


In [6]:
system_df = results_df[results_df["ClassLabel"] == "system"].sort_values("F1Score", ascending=False)
display(system_df)


,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
42,"FLOW, CTLS, STLS, REC",system,0.936765,0.971162,0.936829
39,"CTLS, STLS, REC",system,0.936789,0.971174,0.936327
12,"FLOW, CTLS",system,0.920685,0.965348,0.931180
24,"CTLS, REC",system,0.906104,0.959060,0.927679
33,"FLOW, CTLS, REC",system,0.905903,0.959096,0.927620
36,"FLOW, STLS, REC",system,0.881196,0.940677,0.905187
27,"STLS, REC",system,0.881453,0.941110,0.905024
3,CTLS,system,0.849237,0.921399,0.882392
30,"FLOW, CTLS, STLS",system,0.916458,0.960282,0.868357
21,"CTLS, STLS",system,0.906216,0.955252,0.868314


## Malware Profile

This section isolates the one-class SVM results for the `malware` profile. Compared with the density and reconstruction baselines, this model tests whether a flexible nonlinear boundary around the malware feature space is enough to separate malware traffic from the rest.


In [7]:
malware_df = results_df[results_df["ClassLabel"] == "malware"].sort_values("F1Score", ascending=False)
display(malware_df)


,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
23,"CTLS, STLS",malware,0.801894,0.745785,0.854720
5,CTLS,malware,0.788850,0.810698,0.850664
14,"FLOW, CTLS",malware,0.792042,0.698678,0.850621
32,"FLOW, CTLS, STLS",malware,0.802873,0.703345,0.850112
26,"CTLS, REC",malware,0.823041,0.706219,0.849212
35,"FLOW, CTLS, REC",malware,0.823444,0.706361,0.848260
41,"CTLS, STLS, REC",malware,0.802492,0.702506,0.843604
44,"FLOW, CTLS, STLS, REC",malware,0.802447,0.702068,0.842401
11,REC,malware,0.709418,0.644821,0.784620
20,"FLOW, REC",malware,0.674896,0.573144,0.775574


## Unknown Category

The `unknown` label remains a residual class rather than a clean semantic traffic type, so these results should still be interpreted carefully. The same metric suite is kept here so the one-class SVM baseline can be compared directly with the PCA, Isolation Forest, and GMM notebooks under identical splits and feature subsets.


In [8]:
unknown_df = results_df[results_df["ClassLabel"] == "unknown"].sort_values("F1Score", ascending=False)
display(unknown_df)


,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
1,FLOW,unknown,0.474697,0.842730,0.930261
25,"CTLS, REC",unknown,0.481570,0.863324,0.928987
37,"FLOW, STLS, REC",unknown,0.529940,0.883188,0.928982
22,"CTLS, STLS",unknown,0.515900,0.892683,0.928980
28,"STLS, REC",unknown,0.530751,0.887217,0.928980
4,CTLS,unknown,0.491241,0.866406,0.928980
7,STLS,unknown,0.383855,0.853486,0.928980
34,"FLOW, CTLS, REC",unknown,0.481334,0.860710,0.928980
16,"FLOW, STLS",unknown,0.420519,0.841910,0.928979
13,"FLOW, CTLS",unknown,0.476125,0.853279,0.928968
